## Caption Preprocessing

Di sini, kita preprocess caption pada dataset menggunakan `TextTokenizer` custom yang dibuat di `text_utils.py`. 

In [38]:
import os
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent.parent 
sys.path.append(str(PROJECT_ROOT))

from src.utility.text_utils import TextTokenizer

DATA_DIR = PROJECT_ROOT / "data" / "2_rnn_image_captioning"
CAPTIONS_FILE = DATA_DIR / "captions.txt"
VOCAB_FILE = DATA_DIR / "vocab.json"
OUTPUT_SEQ_FILE = DATA_DIR / "processed_captions.npy"
OUTPUT_IMG_FILE = DATA_DIR / "image_ids.npy"

os.makedirs(DATA_DIR, exist_ok=True)

tokenizer = TextTokenizer()

Baca file `data/2_rnn_image_captioning/captions.txt` dan simpan semua captions yang ada pada dictionary `image_to_captions` dengan key-nya adalah nama file-nya. 

In [39]:
df = pd.read_csv(CAPTIONS_FILE)

image_to_captions = {}
for _, row in df.iterrows():
   img_id = row['image'].split('.')[0]
   caption = row['caption']
   
   if img_id not in image_to_captions:
      image_to_captions[img_id] = []
   image_to_captions[img_id].append(caption)

print(f"Loaded {len(image_to_captions)} unique images.")
print(f"Loaded {len(df)} total captions.")

Loaded 8091 unique images.
Loaded 40455 total captions.


Bangun *vocabulary* menggunakan `.build_vocab()` yang disimpan menjadi dictionary sebagai atribut `tokenizer`. 

In [40]:
all_captions = df['caption'].tolist()
tokenizer.build_vocab(all_captions)

print(f"Built vocabulary. Total unique words: {tokenizer.vocab_size}")

Built vocabulary. Total unique words: 8832


Sekarang, kita konversikan teks caption menjadi *token sequence* dengan `encode()` dan diberikan pad dengan `pad_sequence()`. Hasilnya disimpan pada array `padded_sequences` dengan nama gambar yang berkorespondensi satu-satu dengannya disimpan di array `ordered_image_ids`. 

In [41]:
encoded_sequences = []
ordered_image_ids = []

for img_id, captions in image_to_captions.items():
   for caption in captions:
      seq = tokenizer.encode(caption)
      encoded_sequences.append(seq)
      ordered_image_ids.append(img_id)

max_length = max([len(seq) for seq in encoded_sequences])
padded_sequences = tokenizer.pad_sequence(encoded_sequences, max_length=max_length)

print(f"Total encoded sequences: {len(padded_sequences)}")
print(f"Maximum sequence length: {max_length}")

Total encoded sequences: 40455
Maximum sequence length: 38


Hasil pre-processing disimpan pada disk secara langsung untuk memudahkan: 
- `vocab.json`: menyimpan *vocabulary* semua kata yang ada
- `processed_captions.npy`: menyimpan semua representasi *token sequence* dari caption
- `image_ids.npy`: menyimpan nama file gambar yang berkorespondensi dengan caption
- `config.json`: menyimpan `max_length` dan `vocab_size` untuk memudahkan *look-up*

In [42]:
tokenizer.save_vocab(VOCAB_FILE)
np.save(OUTPUT_SEQ_FILE, padded_sequences)
np.save(OUTPUT_IMG_FILE, np.array(ordered_image_ids))

config = {"max_length": max_length, "vocab_size": tokenizer.vocab_size}
with open(DATA_DIR / "config.json", "w") as f:
   json.dump(config, f)

print(f"Preprocessing complete! All artifacts saved to:\n{DATA_DIR}")

Preprocessing complete! All artifacts saved to:
D:\ITB\Semester 6\ML\Tugas Besar 2\data\2_rnn_image_captioning
